# Drama-3-Japanese TTS Server for stollmly (Google Colab)

[StudioYebisu-ship-it/Drama-3-Japanese-speech-cpp-gguf](https://huggingface.co/StudioYebisu-ship-it/Drama-3-Japanese-speech-cpp-gguf)（ベースモデル: [Aratako/Drama-3-Japanese](https://huggingface.co/Aratako/Drama-3-Japanese)）を [`nyosegawa/speech.cpp`](https://github.com/nyosegawa/speech.cpp) (`v0.11.2`) で起動し、Cloudflare Quick Tunnel 経由で **stollmly** から利用できる OpenAI 互換 Audio API サーバーを立てるノートブックです。

### モデルと VRAM の目安
- **`small-IQ3_XXS` (既定・推奨)**: 約 `9.09 GiB` / 必要空き VRAM 約 `11.48 GiB`。`drama3-speechcpp-fast.patch` 適用済み CUDA ビルドで **Colab 無料 T4 (15 GB) / L4 / A100** で動作します。
- **`Q4_K`**: 約 `20.82 GiB` / 必要空き VRAM 約 `23.21 GiB`。CUDA の FP16 オーバーフローを回避するため **Vulkan / CPU** で実行します（L4 24GB 相当以上推奨）。
- **`Q5_K`**: 約 `24.57 GiB` / 必要空き VRAM 約 `26.96 GiB`。**Vulkan / CPU** で実行します（A100 40GB/80GB 推奨）。

### ライセンス・利用条件
- **Built with Fish Audio.** モデルの重みは **Fish Audio Community License 1.0**（非営利の研究・個人利用向け）に従います。
- `speech.cpp` 用パッチ (`drama3-speechcpp-fast.patch` / `drama3-small-speechcpp.patch`) のライセンス・帰属表記は元リポジトリの `LICENSE-SPEECH-CPP` および `NOTICE` を参照してください。

In [ ]:
#@title 1. GPU 確認とパラメータ設定
import os
import subprocess

MODEL_VARIANT = 'small-IQ3_XXS'  #@param ['small-IQ3_XXS', 'Q4_K', 'Q5_K']
USE_FAST_PATCH = True  #@param {type: 'boolean'}
MAX_PAUSE_FRAMES = 18  #@param {type: 'integer'}
SEED = 42  #@param {type: 'integer'}
USE_GDRIVE_CACHE = False  #@param {type: 'boolean'}

os.environ['SPEECH_DRAMA3_MAX_PAUSE_FRAMES'] = str(MAX_PAUSE_FRAMES)
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

smi = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,compute_cap', '--format=csv,noheader'],
    capture_output=True, text=True
)
if smi.returncode != 0:
    raise RuntimeError('GPU が見つかりません。Colab の「ランタイム → ランタイムのタイプを変更」から GPU を選択してください。')

gpu_info = smi.stdout.strip().splitlines()[0]
gpu_name, gpu_mem_str, compute_cap = [x.strip() for x in gpu_info.split(',')]
gpu_mem_mib = int(gpu_mem_str.split()[0])
print(f'GPU: {gpu_name} ({gpu_mem_mib} MiB, Compute Capability {compute_cap})')
print(f'選択モデル: {MODEL_VARIANT} (USE_FAST_PATCH={USE_FAST_PATCH})')

if MODEL_VARIANT in ('Q4_K', 'Q5_K'):
    print('注意: Q4_K / Q5_K は CUDA ではなく Vulkan (または CPU) で実行します。')
    if MODEL_VARIANT == 'Q4_K' and gpu_mem_mib < 23000:
        print('警告: VRAM が 23 GiB 未満のため、一部レイヤーが CPU にオフロードされるかメモリ不足になる可能性があります。T4 では small-IQ3_XXS を推奨します。')
    if MODEL_VARIANT == 'Q5_K' and gpu_mem_mib < 27000:
        print('警告: VRAM が 27 GiB 未満のため、A100 40GB 以上を推奨します。')

WORK_DIR = '/content/drama3_tts'
MODEL_DIR = os.path.join(WORK_DIR, 'models')
if USE_GDRIVE_CACHE:
    from google.colab import drive
    drive.mount('/content/drive')
    MODEL_DIR = '/content/drive/MyDrive/stollmly_tts_cache'

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print('MODEL_DIR:', MODEL_DIR)

In [ ]:
#@title 2. モデルとパッチを Hugging Face からダウンロード
!pip -q install -U huggingface_hub hf_transfer

from huggingface_hub import hf_hub_download

REPO_ID = 'StudioYebisu-ship-it/Drama-3-Japanese-speech-cpp-gguf'
MODEL_FILES = {
    'small-IQ3_XXS': 'Drama-35B-A3B-Japanese-v3-small-IQ3_XXS.gguf',
    'Q4_K': 'Drama-35B-A3B-Japanese-v3-Q4_K.gguf',
    'Q5_K': 'Drama-35B-A3B-Japanese-v3-Q5_K.gguf',
}

gguf_name = MODEL_FILES[MODEL_VARIANT]
sidecar_name = f'{gguf_name}.json'

MODEL_PATH = hf_hub_download(repo_id=REPO_ID, filename=gguf_name, local_dir=MODEL_DIR)
SIDECAR_PATH = hf_hub_download(repo_id=REPO_ID, filename=sidecar_name, local_dir=MODEL_DIR)
print('GGUF:', MODEL_PATH)
print('Sidecar:', SIDECAR_PATH)

PATCH_PATH = None
if MODEL_VARIANT == 'small-IQ3_XXS':
    patch_name = 'drama3-speechcpp-fast.patch' if USE_FAST_PATCH else 'drama3-small-speechcpp.patch'
    PATCH_PATH = hf_hub_download(repo_id=REPO_ID, filename=patch_name, local_dir=WORK_DIR)
    print('Patch:', PATCH_PATH)

In [ ]:
#@title 3. speech.cpp (v0.11.2) のビルド
import shutil

SPEECH_DIR = os.path.join(WORK_DIR, 'speech.cpp')
if not os.path.exists(SPEECH_DIR):
    !git clone --branch v0.11.2 --depth 1 --recurse-submodules https://github.com/nyosegawa/speech.cpp.git {SPEECH_DIR}

%cd {SPEECH_DIR}
if PATCH_PATH:
    check = subprocess.run(['git', 'apply', '--check', PATCH_PATH], capture_output=True, text=True)
    if check.returncode == 0:
        !git apply {PATCH_PATH}
        print('パッチを適用しました:', PATCH_PATH)
    else:
        print('パッチは既に適用済みかスキップされました。')

USE_VULKAN = MODEL_VARIANT in ('Q4_K', 'Q5_K')
VULKAN_AVAILABLE = False

if not USE_VULKAN:
    arch = compute_cap.replace('.', '')
    !cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON -DGGML_CUDA_GRAPHS=ON -DCMAKE_CUDA_ARCHITECTURES={arch}
    !cmake --build build --target speech speech-cli -j$(nproc)
else:
    !apt-get -qq update && apt-get -qq install -y libvulkan-dev vulkan-tools glslc
    !cmake -S . -B build -DCMAKE_BUILD_TYPE=Release -DGGML_VULKAN=ON
    !cmake --build build --target speech speech-cli -j$(nproc)
    dev_out = subprocess.run(['./build/speech', 'devices'], capture_output=True, text=True).stdout
    print(dev_out)
    VULKAN_AVAILABLE = 'Vulkan0' in dev_out
    if not VULKAN_AVAILABLE:
        print('注意: この Colab ランタイムでは Vulkan0 が検出されませんでした。CPU フォールバックで実行されます。')

SPEECH_BIN = os.path.join(SPEECH_DIR, 'build', 'speech')
assert os.path.exists(SPEECH_BIN), 'speech バイナリのビルドに失敗しました'
print('ビルド完了:', SPEECH_BIN)

In [ ]:
#@title 4. (任意) 参照音声からボイスクローン用ファイルを作成
#@markdown 参照音声 (.wav) を使う場合は `ENABLE_VOICE_CLONE` をオンにし、正確な書き起こしを入力してください。
ENABLE_VOICE_CLONE = False  #@param {type: 'boolean'}
REF_TRANSCRIPT = '今日はいい天気ですね。'  #@param {type: 'string'}
VOICE_NAME = 'custom'  #@param {type: 'string'}

VOICE_GGUF_PATH = os.path.join(WORK_DIR, f'{VOICE_NAME}.voice.gguf')
if ENABLE_VOICE_CLONE:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('音声ファイルがアップロードされませんでした。')
    ref_filename = next(iter(uploaded.keys()))
    ref_path = os.path.join(SPEECH_DIR, ref_filename)
    extra_args = ['--device', 'Vulkan0'] if (USE_VULKAN and VULKAN_AVAILABLE) else []
    env = os.environ.copy()
    if USE_VULKAN:
        env['GGML_VK_DISABLE_F16'] = '1'
        env['GGML_VK_DISABLE_COOPMAT'] = '1'
        env['GGML_VK_DISABLE_COOPMAT2'] = '1'
    cmd = [SPEECH_BIN, 'voice', MODEL_PATH, ref_path, '--transcript', REF_TRANSCRIPT, *extra_args, VOICE_GGUF_PATH]
    subprocess.run(cmd, env=env, check=True)
    print('ボイスファイルを作成しました:', VOICE_GGUF_PATH)
else:
    print('スキップしました (既定の --voice none を使用します)。')

In [ ]:
#@title 5. 単発音声生成テスト (インライン再生)
from IPython.display import Audio, display

TEST_TEXT = '[囁き] ねえ、少しだけ声を落として。誰にも聞こえないように、今の話を続けよう。'  #@param {type: 'string'}
test_out = os.path.join(WORK_DIR, 'test_out.wav')
voice_arg = VOICE_GGUF_PATH if os.path.exists(VOICE_GGUF_PATH) else 'none'

cmd = [SPEECH_BIN, 'tts', MODEL_PATH, '--voice', voice_arg, '--seed', str(SEED), '-o', test_out]
if MODEL_VARIANT == 'small-IQ3_XXS':
    cmd.append('--experts-on-cpu=false')
elif USE_VULKAN and VULKAN_AVAILABLE:
    cmd.extend(['--device', 'Vulkan0'])
cmd.append(TEST_TEXT)

env = os.environ.copy()
if USE_VULKAN:
    env['GGML_VK_DISABLE_F16'] = '1'
    env['GGML_VK_DISABLE_COOPMAT'] = '1'
    env['GGML_VK_DISABLE_COOPMAT2'] = '1'

subprocess.run(cmd, env=env, check=True)
display(Audio(test_out, autoplay=False))

In [ ]:
#@title 6. speech serve + Cloudflare Quick Tunnel を起動 (stollmly 接続用 URL 発行)
import re
import time
import urllib.request

CLOUDFLARED_BIN = '/usr/local/bin/cloudflared'
if not os.path.exists(CLOUDFLARED_BIN):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O {CLOUDFLARED_BIN}
    !chmod +x {CLOUDFLARED_BIN}

serve_cmd = [SPEECH_BIN, 'serve', MODEL_PATH, '--host', '127.0.0.1', '--port', '8080', '--cors-origin', '*']
if MODEL_VARIANT == 'small-IQ3_XXS':
    serve_cmd.append('--experts-on-cpu=false')
elif USE_VULKAN and VULKAN_AVAILABLE:
    serve_cmd.extend(['--device', 'Vulkan0'])
if os.path.exists(VOICE_GGUF_PATH):
    serve_cmd.extend(['--add-voice', f'{VOICE_NAME}={VOICE_GGUF_PATH}'])

env = os.environ.copy()
if USE_VULKAN:
    env['GGML_VK_DISABLE_F16'] = '1'
    env['GGML_VK_DISABLE_COOPMAT'] = '1'
    env['GGML_VK_DISABLE_COOPMAT2'] = '1'

server_proc = subprocess.Popen(serve_cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print('speech serve を起動しています (モデル読み込み完了まで待機)...')

ready = False
for _ in range(180):
    if server_proc.poll() is not None:
        out = server_proc.stdout.read() if server_proc.stdout else ''
        raise RuntimeError(f'speech serve が異常終了しました:\n{out}')
    try:
        with urllib.request.urlopen('http://127.0.0.1:8080/health', timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    server_proc.terminate()
    raise TimeoutError('speech serve の起動がタイムアウトしました。')

print('speech serve の準備が完了しました。Cloudflare Quick Tunnel を起動します...')
tunnel_proc = subprocess.Popen(
    [CLOUDFLARED_BIN, 'tunnel', '--url', 'http://127.0.0.1:8080'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

public_url = None
url_pattern = re.compile(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com')
try:
    for line in tunnel_proc.stdout:
        m = url_pattern.search(line)
        if m and public_url is None:
            public_url = m.group(0)
            print('=' * 64)
            print('stollmly の「設定 → 音声合成 (TTS) → TTS サーバー URL」に以下を入力してください:')
            print(f'  {public_url}')
            if os.path.exists(VOICE_GGUF_PATH):
                print(f'ボイス名: {VOICE_NAME} (既定は none)')
            print('停止するにはこのセルを中断 (■) してください。')
            print('=' * 64)
except KeyboardInterrupt:
    print('サーバーを停止しています...')
finally:
    tunnel_proc.terminate()
    server_proc.terminate()
    tunnel_proc.wait(timeout=5)
    server_proc.wait(timeout=5)